In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

### **Data Reading**

In [0]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@tjdatabricksete.dfs.core.windows.net/products")

In [0]:
display(df)

In [0]:
df = df.drop("_rescued_data")

In [0]:
df.createOrReplaceTempView("products")

### **Functions**

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricksete_cat.bronze.discount_func(p_price DOUBLE)
RETURNS DOUBLE
RETURN p_price * 0.90

In [0]:
%sql
select product_id, databricksete_cat.bronze.discount_func(price) as discounted_price , price
from products

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricksete_cat.bronze.upper_func(p_brand STRING)
RETURNS STRING
LANGUAGE PYTHON
AS
$$
    return p_brand.upper()
$$
    

In [0]:
%sql
select product_id, databricksete_cat.bronze.upper_func(brand) as brand_upper
from products

In [0]:
df = df.withColumn("discounted_price",expr("databricksete_cat.bronze.discount_func(price)"))
df.display()

In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@tjdatabricksete.dfs.core.windows.net/products")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricksete_cat.silver.products_silver
USING DELTA
LOCATION 'abfss://silver@tjdatabricksete.dfs.core.windows.net/products'